# Click prompts collector — whip videos

Runs on **olab-1's local Jupyter** (the one you already have at `http://127.0.0.1:8888/`). 
Frames stay on bigpurple. For each video we:

1. Ask bigpurple how many frames the video has (one ssh ls).
2. Rsync **just the 3 prompt frames** we need to click on (~5 MB per video).
3. You click. Left = positive, right = negative. Number keys 1/2/3 switch object id.
4. Save `prompts/<video>.json` locally and push it to bigpurple.

When you're done, on bigpurple run `sbatch --array=0-$((N-1))%8 run_inference_array.sh` to fan the inference out.

In [ ]:
%matplotlib widget
from pathlib import Path
import json, subprocess, os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

BP_HOST      = 'bigpurple'           # ssh alias; only used when running off-cluster
BP_FRAMES    = '/gpfs/data/oermannlab/private_data/whip/frames_attempt2'
BP_REPO      = '/gpfs/data/oermannlab/users/schula12/Surgical-SAM-2'

# Detect once: if the GPFS path is reachable directly (we're on bigpurple),
# we can skip the ssh+rsync round-trip and point matplotlib straight at the
# source files. Saves time and avoids "image file is truncated" issues from
# partial rsync transfers back into the same filesystem.
ON_BIGPURPLE = Path(BP_FRAMES).is_dir()
print(f'ON_BIGPURPLE = {ON_BIGPURPLE}')

CACHE_DIR    = Path('test_data/_prompt_frames')   # only used off-cluster
CACHE_DIR.mkdir(parents=True, exist_ok=True)

PROMPTS_DIR  = Path('prompts')
PROMPTS_DIR.mkdir(exist_ok=True)

N_PROMPT_FRAMES = 3

OBJ_COLORS = ['#00ff00', '#ff8800', '#00bfff', '#ff00ff', '#ffff00',
              '#ff0000', '#00ffff', '#ffffff']

def _strip_banner(out):
    return '\n'.join(l for l in out.splitlines()
                     if 'Loading' not in l and 'requirement' not in l)

def ssh(cmd):
    """Run a command on bigpurple (off-cluster) OR locally (on-cluster)."""
    if ON_BIGPURPLE:
        r = subprocess.run(['bash', '-lc', cmd], capture_output=True, text=True)
    else:
        r = subprocess.run(['ssh', BP_HOST, cmd], capture_output=True, text=True)
    if r.returncode != 0:
        print('stderr:', r.stderr)
        r.check_returncode()
    return _strip_banner(r.stdout.strip())

def list_videos():
    """Return [(video_name, frame_count), ...] for videos with >=3 frames."""
    if ON_BIGPURPLE:
        out = []
        for d in sorted(Path(BP_FRAMES).iterdir()):
            if not d.is_dir(): continue
            if d.name.startswith('480full'): continue
            try:
                n = sum(1 for _ in d.iterdir())
            except OSError:
                continue
            if n >= 3:
                out.append((d.name, n))
        return out
    raw = ssh(f'for d in {BP_FRAMES}/*/; do '
              f'  n=$(ls $d 2>/dev/null | wc -l); '
              f'  printf "%s %d\\n" "$(basename $d)" $n; '
              f'done')
    out = []
    for line in raw.splitlines():
        parts = line.strip().rsplit(' ', 1)
        if len(parts) != 2: continue
        name, n_s = parts
        if name.startswith('480full'): continue
        try:
            n = int(n_s)
        except ValueError:
            continue
        if n >= 3:
            out.append((name, n))
    return out

videos = list_videos()
print(f'{len(videos)} videos with >=3 frames')
for v, n in videos[:5]:
    print(f'  {v}: {n} frames')
if len(videos) > 5:
    print(f'  ... and {len(videos)-5} more')

In [ ]:
def prompt_frame_indices(n_total, n_prompt=N_PROMPT_FRAMES):
    return [int(i) for i in np.linspace(0, n_total-1, n_prompt, dtype=int)]

def fetch_prompt_frames(video_name, n_total):
    """Return [(loader_idx, local_path), ...] for the 3 prompt frames.
    On bigpurple: point directly at the GPFS source (no copy).
    Off-cluster: ssh+ls to learn filenames, rsync each to a local cache."""
    idxs = prompt_frame_indices(n_total)

    if ON_BIGPURPLE:
        src_dir = Path(BP_FRAMES) / video_name
        all_files = sorted(p.name for p in src_dir.iterdir())
        assert len(all_files) == n_total, (
            f'{video_name}: expected {n_total} files, found {len(all_files)} on GPFS'
        )
        return [(i, src_dir / all_files[i]) for i in idxs]

    # Off-cluster (e.g. olab-1): ssh list, then rsync the chosen frames.
    raw = ssh(f'cd {BP_FRAMES}/{video_name} && ls | sort')
    all_files = raw.splitlines()
    assert len(all_files) == n_total, (
        f'{video_name}: expected {n_total} files, ssh ls returned {len(all_files)}'
    )
    chosen = [(i, all_files[i]) for i in idxs]
    local_dir = CACHE_DIR / video_name
    local_dir.mkdir(parents=True, exist_ok=True)
    local_paths = []
    for i, fname in chosen:
        target = local_dir / f'idx{i:07d}__{fname}'
        if not target.exists():
            subprocess.run(
                ['rsync', '-az',
                 f'{BP_HOST}:{BP_FRAMES}/{video_name}/{fname}', str(target)],
                check=True, capture_output=True)
        local_paths.append((i, target))
    return local_paths

def first_undone():
    for v, n in videos:
        if not (PROMPTS_DIR / f'{v}.json').exists():
            return v, n
    return None, None

_v, _n = first_undone()
if _v:
    print(f'Next un-done: {_v} ({_n} frames). Run the click cell below.')
else:
    print('Every video already has a prompts JSON. Nothing to do.')

In [ ]:
import io
from PIL import ImageFile as _ImageFile
from matplotlib.widgets import Button as _MplButton
_ImageFile.LOAD_TRUNCATED_IMAGES = True

def _safe_open(image_path):
    """Read the whole file first, then decode. Avoids PIL streaming-truncation on GPFS."""
    with open(image_path, 'rb') as fp:
        data = fp.read()
    img = Image.open(io.BytesIO(data))
    img.load()
    return img


class ClickCollector:
    """Walks a single video's prompt frames inside ONE figure with a 'Next' button.

    Usage:
        col = ClickCollector(video_name, n_total)
        col.start()   # opens the figure
        # user clicks on the image, presses 1/2/3 to switch object,
        # clicks the on-figure 'Next frame >>' button to advance.
        # After the last frame, the JSON saves automatically.
    """

    def __init__(self, video_name, n_total, push=True):
        self.video_name = video_name
        self.n_total = n_total
        self.push = push
        self.chosen = fetch_prompt_frames(video_name, n_total)   # [(loader_idx, path), ...]
        first = _safe_open(self.chosen[0][1])
        self.W, self.H = first.size
        self.step = 0
        self.current_obj = 1
        self.current_clicks = {}       # {obj_id: {'positive':[], 'negative':[]}}
        self.objects_by_frame = {}     # {loader_idx: [{obj_id,positive,negative}]}
        self.fig = None
        self.ax = None
        self.btn = None
        self.info = None

    def start(self):
        self.fig, self.ax = plt.subplots(figsize=(13, 8))
        plt.subplots_adjust(bottom=0.12)
        # Next button axes (bottom-center)
        btn_ax = self.fig.add_axes([0.42, 0.02, 0.16, 0.055])
        self.btn = _MplButton(btn_ax, 'Next frame  ➔')
        self.btn.on_clicked(self._on_next)
        self.fig.canvas.mpl_connect('button_press_event', self._on_click)
        self.fig.canvas.mpl_connect('key_press_event', self._on_key)
        self._render()
        plt.show()

    def _render(self):
        loader_idx, path = self.chosen[self.step]
        img = _safe_open(path)
        self.ax.clear()
        self.ax.imshow(img)
        self.ax.axis('off')
        self.ax.set_title(
            f'{self.video_name}  —  prompt {self.step+1}/{len(self.chosen)}  '
            f'(loader idx {loader_idx})', fontsize=11)
        self.current_clicks = {}
        self.current_obj = 1
        # status overlay
        if self.info is not None:
            try: self.info.remove()
            except Exception: pass
        self.info = self.ax.text(
            0.01, 0.99, '', transform=self.ax.transAxes, ha='left', va='top',
            fontsize=11, color='black')
        self._refresh_info()
        self.fig.canvas.draw_idle()

    def _refresh_info(self):
        color = OBJ_COLORS[(self.current_obj-1) % len(OBJ_COLORS)]
        self.info.set_text(
            f'obj {self.current_obj}  ·  Left=positive  Right=negative  ·  '
            f'press 1/2/3 to switch  ·  click "Next frame" when done')
        self.info.set_bbox(dict(facecolor=color, alpha=0.85, edgecolor='none'))

    def _on_click(self, event):
        if event.inaxes != self.ax or event.xdata is None: return
        oid = self.current_obj
        self.current_clicks.setdefault(oid, {'positive': [], 'negative': []})
        color = OBJ_COLORS[(oid-1) % len(OBJ_COLORS)]
        if event.button == 1:
            self.current_clicks[oid]['positive'].append(
                [round(event.xdata, 1), round(event.ydata, 1)])
            self.ax.plot(event.xdata, event.ydata, marker='+', color=color, ms=22, mew=3)
        elif event.button == 3:
            self.current_clicks[oid]['negative'].append(
                [round(event.xdata, 1), round(event.ydata, 1)])
            self.ax.plot(event.xdata, event.ydata, marker='x', color=color, ms=22, mew=3)
        self.fig.canvas.draw_idle()

    def _on_key(self, event):
        if event.key and event.key.isdigit() and event.key != '0':
            self.current_obj = int(event.key)
            self._refresh_info()
            self.fig.canvas.draw_idle()

    def _on_next(self, _evt):
        # Persist clicks from current frame.
        loader_idx = self.chosen[self.step][0]
        if self.current_clicks:
            self.objects_by_frame[int(loader_idx)] = [
                {'obj_id': oid,
                 'positive': v['positive'],
                 'negative': v['negative']}
                for oid, v in sorted(self.current_clicks.items())
            ]
        self.step += 1
        if self.step >= len(self.chosen):
            self._save()
            plt.close(self.fig)
        else:
            self._render()

    def _save(self):
        out = {
            'video': self.video_name,
            'resolution': [self.W, self.H],
            'n_frames': self.n_total,
            'prompt_frames': [i for i, _ in self.chosen],
            'objects_by_frame': {str(k): v for k, v in self.objects_by_frame.items()},
        }
        out_path = PROMPTS_DIR / f'{self.video_name}.json'
        with open(out_path, 'w') as fp:
            json.dump(out, fp, indent=2)
        print(f'Saved {out_path}')
        if self.push and not ON_BIGPURPLE:
            subprocess.run(
                ['rsync', '-az', str(out_path), f'{BP_HOST}:{BP_REPO}/prompts/'],
                check=True)
            print(f'Pushed to {BP_HOST}:{BP_REPO}/prompts/{out_path.name}')


def collect_for_video(video_name, n_total, push=True):
    """Convenience wrapper that creates a collector and starts it."""
    col = ClickCollector(video_name, n_total, push=push)
    col.start()
    return col

print('Helpers loaded.  ON_BIGPURPLE =', ON_BIGPURPLE)

## Click one video

In [ ]:
VIDEO = 'DG_whip_16598313'   # <-- pick from the list printed above
n_total = dict(videos)[VIDEO]
out = collect_for_video(VIDEO, n_total)
print(json.dumps(out, indent=2)[:600])

## Walk every un-done video

In [ ]:
LIMIT = 5   # set to None for all of them
done = 0
for vname, n_total in videos:
    if (PROMPTS_DIR / f'{vname}.json').exists():
        continue
    collect_for_video(vname, n_total)
    done += 1
    if LIMIT is not None and done >= LIMIT:
        break
n_done = sum(1 for v,_ in videos if (PROMPTS_DIR/(v+'.json')).exists())
print(f'\nThis session: {done} videos. Total done: {n_done} / {len(videos)}.')